In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve

In [ ]:
train=pd.read_csv('/kaggle/input/playground-series-s3e23/train.csv')
test=pd.read_csv('/kaggle/input/playground-series-s3e23/test.csv')

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
train.isnull().sum()

In [ ]:
test.isnull().sum()

In [ ]:
train.info()

In [ ]:
test.info()

In [ ]:
train=train.drop('id',axis=1)
X_train=train.drop('defects',axis=1)
y_train=train['defects']

In [ ]:
scaler=MinMaxScaler()
X_train=scaler.fit_transform(X_train)

In [ ]:
plt.figure(figsize=(12,10))
cor = train.corr()
sns.heatmap(cor, annot=True, cmap=plt.cm.Reds)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
train['defects'].value_counts().plot.pie(explode=[0,0.1], autopct='%1.1f%%', shadow=True)
ax.set_title('defects')
plt.show()

In [ ]:
rf=RandomForestClassifier(n_estimators=100,max_depth=5,max_features=1,random_state=42)
rf.fit(X_train,y_train)

y_pred_rf=rf.predict(X_train)
print('Accuracy:',accuracy_score(y_train,y_pred_rf))
print('Classification Report:',classification_report(y_train,y_pred_rf))
print('Confusion Matrix:',confusion_matrix(y_train,y_pred_rf))
print('ROC AUC Score:',roc_auc_score(y_train,y_pred_rf))

fpr, tpr, thresholds = roc_curve(y_train, y_pred_rf)
plt.plot([0,1],[0,1],'k--')
plt.plot(fpr,tpr,label='Random Forest')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Random Forest ROC Curve')
plt.show()

In [ ]:
xg=XGBClassifier(n_estimators=100,max_depth=5,min_child_weight=1,max_delta_step=0,random_state=42)
xg.fit(X_train,y_train)

y_pred_xg=xg.predict(X_train)
print('Accuracy:',accuracy_score(y_train,y_pred_xg))
print('Classification Report:',classification_report(y_train,y_pred_xg))
print('Confusion Matrix:',confusion_matrix(y_train,y_pred_xg))

fpr, tpr, thresholds = roc_curve(y_train, y_pred_xg)
plt.plot([0,1],[0,1],'k--')
plt.plot(fpr,tpr,label='XGBoost')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('XGBoost ROC Curve')
plt.show()

In [ ]:
lg=LGBMClassifier(n_estimators=100,force_col_wise=True,random_state=42)
lg.fit(X_train,y_train)

y_pred_lg=lg.predict(X_train)
print('Accuracy:',accuracy_score(y_train,y_pred_lg))
print('Classification Report:',classification_report(y_train,y_pred_lg))
print('Confusion Matrix:',confusion_matrix(y_train,y_pred_lg))

fpr, tpr, thresholds = roc_curve(y_train, y_pred_lg)
plt.plot([0,1],[0,1],'k--')
plt.plot(fpr,tpr,label='LightGBM')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('LightGBM ROC Curve')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20, 5))
sns.heatmap(confusion_matrix(y_train,y_pred_rf), annot=True, cmap='viridis', ax=ax[0])
ax[0].set_title('Random Forest')
sns.heatmap(confusion_matrix(y_train,y_pred_xg), annot=True, cmap='viridis', ax=ax[1])
ax[1].set_title('XGBoost')
sns.heatmap(confusion_matrix(y_train,y_pred_lg), annot=True, cmap='viridis', ax=ax[2])
ax[2].set_title('LightGBM')
plt.show()

In [ ]:
y_test=xg.predict(test.drop('id',axis=1))
submission=pd.DataFrame({'id':test['id'],'defects':y_test})
submission.to_csv('submission.csv',index=False)